# 1.并查集

## 1.1带权DSU

In [ ]:
# 可以看食物链这道题

N = int(2e5 + 10)
sys.setrecursionlimit(N)
dx, dy = [0, 1, 0, -1, 1, -1, 1, -1], [1, 0, -1, 0, -1, -1, 1, 1]
inf = float('inf')
n, q = MII()
p = [i for i in range(N)]
dis = [0] * (N)
def find(x):
    if x != p[x]:
        t = p[x]
        p[x] = find(p[x]) 
        dis[x] += dis[t]  # 自己到父结点的距离+父结点到根的距离（递归）=自己到根的距离
    return p[x]
cnt = 0
res = []
for i in range(1, q + 1):
    x, y, w = MII()
    a, b = find(x), find(y)
    if a != b or dis[x] - dis[y] == w:
        res.append(i)
        p[a] = b
        dis[a] = dis[y] - dis[x] + w  # 更新距离
print(*res)



## 1.2DSU

In [ ]:

# 递归实现

p = [i for i in range(n + 1)]

def find(x):
    if x != p[x]:
        p[x] = find(p[x])
    return p[x]




## 迭代实现


def find(x):
    t = x
    while x != p[x]:
        x = p[x]
    while t != x:
        p[t], t = x, p[t]
    return x


# 2. 单调队列

## 2.1长为k的窗口最大值

In [ ]:
from collections import deque
class MyQueue: #单调队列（从大到小
    def __init__(self):
        self.queue = deque() #使用list来实现单调队列
    
    #每次弹出的时候，比较当前要弹出的数值是否等于队列出口元素的数值，如果相等则弹出。
    #同时pop之前判断队列当前是否为空。
    def pop(self, value):
        if self.queue and value == self.queue[0]:
            self.queue.popleft()#list.pop()时间复杂度为O(n),这里可以使用collections.deque()
            
    #如果push的数值大于入口元素的数值，那么就将队列后端的数值弹出，直到push的数值小于等于队列入口元素的数值为止。
    #这样就保持了队列里的数值是单调从大到小的了。
    def push(self, value):
        while self.queue and value > self.queue[-1]:
            self.queue.pop()
        self.queue.append(value)
        
    #查询当前队列里的最大值 直接返回队列前端也就是front就可以了。
    def front(self):
        return self.queue[0]
    

"""
# LC 239. 滑动窗口最大值
class Solution:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:
        q = deque()
        res = []
        for i, x in enumerate(nums):
            while q and nums[q[-1]] <= x:
                q.pop()
            q.append(i)
            while q and i - q[0] + 1 > k:
                q.popleft()
            if i >= k - 1:
                res.append(nums[q[0]])
        return res

"""

# 3.单调栈

## 3.1子数组的最小值的和（贡献法）

In [ ]:
# LC2104. 子数组范围和（单调栈的应用）
from typing import List
class Solution:
    def subArrayRanges(self, nums: List[int]) -> int:
        def sumSubarrayMaxs(arr: List[int]) -> int:  # 计算每个数字作为最大值的贡献
            n = len(arr)
            left, right = [-1] * n, [n] * n
            st = []
            for i, x in enumerate(arr):
                while st and arr[st[-1]] <= x:  # 左边是大于   
                    st.pop()
                if st:
                    left[i] = st[-1]
                st.append(i)
            st = []
            for i in range(n - 1, -1, -1):
                while st and arr[st[-1]] < arr[i]:  # 右边是大于等于， 这样做是为了防止重复计算
                    st.pop()
                if st:
                    right[i] = st[-1]
                st.append(i)
            res = 0
            for i, x in enumerate(arr):
                res += x * (i - left[i]) * (right[i] - i)
            return res
        r1 = sumSubarrayMaxs(nums)
        nums = [-x for x in nums]  # 计算小技巧，数组取反就是最小值的贡献
        r2 = sumSubarrayMaxs(nums)
        return r1 + r2
    

# 或者把两次循坏改成一次循环
"""
注意到在计算 left的过程中,如果栈顶元素 ≥arr[i]，那么 i 就是栈顶元素的右边界，因此前两个循环可以合并。
更详细的解释：对于栈顶元素 t, 如果 t 右侧有多个小于或等于 t 的元素，那么 t 只会因为右侧第一个小于或等于 t 的元素而出栈，这恰好符合右边界的定义。
"""
def sumSubarrayMaxs(arr: List[int]) -> int:  # 计算每个数字作为最大值的贡献
    n = len(arr)
    left, right = [-1] * n, [n] * n
    st = []
    for i, x in enumerate(arr):
        while st and arr[st[-1]] <= x:  # 左边是大于   
            right[st.pop()] = i   # # i恰好是栈顶的右边界
        if st:
            left[i] = st[-1]
        st.append(i)
    res = 0
    for i, x in enumerate(arr):
        res += x * (i - left[i]) * (right[i] - i)
    return res

# 4.线段树

## 4.1单点修改

In [ ]:
# 把区间表示为若干区间的并集
class SegmentTree:
    def __init__(self, nums) -> None:
        self.nums = nums
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)
        self._build(1, 1, self.n)

    def _build(self, o, l, r):
        if l == r:
            self.tree[o] = self.nums[l - 1] # 数据的下标从1开始，线段树也是从1开始。直接传入数组，不要加[0]
            return
        mid = l + r >> 1
        self._build(o * 2, l, mid)
        self._build(o * 2 + 1, mid + 1, r)
        self.tree[o] = self.tree[o * 2] + self.tree[o * 2 + 1]
    
    def _query(self, o, l, r, L, R):  # 求[L, R] 范围内的元素和
        if L <= l and r <= R:
            return self.tree[o]
        tree = 0
        mid = (l + r) // 2
        if L <= mid:
            tree += self._query(o * 2, l, mid, L, R)
        if R > mid:
            tree += self._query(o * 2 + 1, mid + 1, r, L, R)
        return tree
    
    def _update(self, o, l, r, idx, val):  # 当前结点和它表示的区间, 需要给idx对应下标+val
        if l == r:  # 递归到了叶子，一定等于idx，直接加
            self.tree[o] += val
            return
        mid = (l + r) // 2
        if idx <= mid:  # 往左子树递归
            self._update(o * 2, l, mid, idx, val)
        else:  # 往左子树递归
            self._update(o * 2 + 1, mid + 1, r, idx, val)
        self.tree[o] = self.tree[o * 2] + self.tree[o * 2 + 1]  # 维护上面的区间

    def query(self, l, r):
        return self._query(1, 1, self.n, l, r)

    def update(self, idx, val):
        return self._update(1, 1, self.n, idx, val)
        
        

## 4.2动态开点

In [ ]:
from bisect import bisect
from typing import List


# LC2736
class Node:
    __slots__ = "l", "r", "mid", "val", "left", "right"

    def __init__(self, l: int, r: int):
        self.l = l
        self.r = r
        self.mid = (l + r) >> 1
        self.val = -1
        self.left = None
        self.right = None


def pushUp(node: Node):
    node.val = max(node.left.val, node.right.val)


def pushDown(node: Node):
    if node.left is None:
        node.left = Node(node.l, node.mid)
    if node.right is None:
        node.right = Node(node.mid + 1, node.r)


class Seg:
    def __init__(self):
        self.root = Node(0, int(1e9))

    def update(self, x, val, node: 'Node' = None):
        if node is None:
            node = self.root
        if node.l == x and node.r == x:
            node.val = max(node.val, val)
            return
        pushDown(node)
        if x <= node.mid:
            self.update(x, val, node.left)
        else:
            self.update(x, val, node.right)
        pushUp(node)

    def query(self, l, r, node: 'Node' = None):
        if node is None:
            node = self.root
        if l <= node.l and node.r <= r:
            return node.val
        pushDown(node)
        ans = -1
        if l <= node.mid:
            ans = max(ans, self.query(l, r, node.left))
        if r > node.mid:
            ans = max(ans, self.query(l, r, node.right))
        return ans


class Solution:
    def maximumSumQueries(self, nums1: List[int], nums2: List[int], q: List[List[int]]) -> List[int]:
        n = len(nums1)
        s = []
        for i in range(n):
            s.append([nums1[i], nums2[i], nums1[i] + nums2[i]])
        s.sort()
        seg = Seg()
        r = n - 1
        for i in range(len(q)):
            q[i].append(i)
        q.sort(reverse=True)
        ans = [-1] * len(q)
        for x, y, i in q:
            k = bisect.bisect_left(s, [x, -1, -1])
            if k == n:                              #没有nums1满足nums1[i]>=x
                continue
            while r >= k:
                seg.update(s[r][1], s[r][2])
                r -= 1
            ans[i] = seg.query(y, int(1e9))
        return ans




## 4.3区间修改

In [ ]:

class LazySegmentTree:
    
    def __init__(self, nums) -> None:
        self.n = len(nums)
        N = 4 * self.n
        self.nums = nums
        self.lazy = [0] * N
        self.sum = [0] * N
        self.build(1, 1, self.n)
        

    def pushup(self, o):  # 子节点更新父结点
        self.sum[o] = self.sum[o * 2] + self.sum[o * 2 + 1]

    def pushdown(self, o, l, r):
        if self.lazy[o]:
            self.lazy[o * 2] += self.lazy[o]    # 懒标记传递给子节点
            self.lazy[o * 2 + 1] += self.lazy[o]
            mid = l + r >> 1
            self.sum[o * 2] += (mid - l + 1) * self.lazy[o]  # 维护子节点的区间和
            self.sum[o * 2 + 1] += (r - mid) * self.lazy[o]
            self.lazy[o] = 0      # 清除标记

    def build(self, o, l, r):
        if l == r:
            self.sum[o] = self.nums[l - 1] # 传进来的数据是从1开始的，所以要减1。根据数据更改
            return
        mid = l + r >> 1
        self.build(o * 2, l, mid)
        self.build(o * 2 + 1, mid + 1, r)
        self.pushup(o) # 更新结点信息

    def _update(self, o, l, r, L, R, val):
        if l >= L and r <= R:
            self.sum[o] += (r - l + 1) * val
            self.lazy[o] += val
            return
        self.pushdown(o, l, r)   # 只要有分裂就要pushdown，保证lazy标记的一致性，否则会出错
        mid = l + r >> 1
        if L <= mid:
            self._update(o * 2, l, mid, L, R, val)
        if R > mid:
            self._update(o * 2 + 1, mid + 1, r, L, R, val)
        self.pushup(o)

    def _query(self,o, l, r, L, R):
        res = 0
        if l >= L and r <= R:
            return self.sum[o]
        self.pushdown(o, l, r)  
        mid = l + r >> 1
        if L <= mid:
            res += self._query(o * 2, l, mid, L, R)
        if R > mid:
            res += self._query(o * 2 + 1, mid + 1, r, L, R)
        return res

    def query(self, l, r):
        return self._query(1, 1, self.n, l, r)

    def update(self, l, r, val):
        return self._update(1, 1, self.n, l, r, val)




## 4.4扫描线

In [ ]:
class TreeNode():
    def __init__(self):
        self.l = 0
        self.r = 0
        self.cnt = 0
        self.length = 0

def build(u, l, r):
    tr[u].l = l
    tr[u].r = r
    if l == r:
        return
    mid = (l + r) // 2
    build(u * 2, l, mid)
    build(u * 2 + 1, mid + 1, r)

def pushup(u):
    if tr[u].cnt:
        l, r = tr[u].l, tr[u].r
        tr[u].length = s[r + 1] - s[l]  # 向右偏移一位
    elif tr[u].l != tr[u].r:
        tr[u].length = tr[u * 2].length + tr[u * 2 + 1].length  # 左右儿子更新
    else:
        tr[u].length = 0

def modify(u, l, r, v):
    if tr[u].l >= l and tr[u].r <= r:  # 包含
        tr[u].cnt += v
    else:
        mid = (tr[u].l + tr[u].r) // 2
        if l <= mid:
            modify(u * 2, l, r, v)
        if r > mid:
            modify(u * 2 + 1, l, r, v)
    pushup(u)  # 更新

T = 1
while T:
    n = II()
    if not n:
        break
    edges = []
    s = set()
    for _ in range(n):
        x1, y1, x2, y2 = list(map(float, input().split()))
        s.add(y1)
        s.add(y2)
        edges.append((x1, y1, y2, 1))
        edges.append((x2, y1, y2, -1))
    s = sorted(s)  # 离散化y坐标
    edges.sort()  # 按x坐标排序
    m = len(s)
    mp = {x: i for i, x in enumerate(s)}
    tr = [TreeNode() for _ in range(m * 4)]
    build(1, 0, m)
    x1 = res = 0
    for x2, y1, y2, c in edges:
        res += tr[1].length * (x2 - x1)
        modify(1, mp[y1], mp[y2] - 1, c)  # 向左偏移一个
        x1 = x2
    print('Test case #%d' %T)
    print('Total explored area: %.2f' %res) 
    print()
    T += 1



## 4.5主席树

In [ ]:
import bisect


N = 100010

class Node:
    __slots__ = "l", "r", "cnt"

    def __init__(self):
        self.l = 0
        self.r = 0
        self.cnt = 0

def build(l, r):
    global idx
    idx += 1
    p = idx
    if l == r: 
        return p
    mid = l + r >> 1
    tree[p].l = build(l, mid)
    tree[p].r = build(mid+1, r)
    return p

def insert(p, l, r, x):
    global idx
    idx += 1
    q = idx
    tree[q].l, tree[q].r, tree[q].cnt = tree[p].l, tree[p].r, tree[p].cnt
    if l == r:
        tree[q].cnt += 1
        return q
    mid = l + r >> 1
    if x <= mid: 
        tree[q].l = insert(tree[p].l, l, mid, x)
    else: 
        tree[q].r = insert(tree[p].r, mid+1, r, x)
    tree[q].cnt = tree[tree[q].l].cnt + tree[tree[q].r].cnt
    return q
    
def query(q, p, l, r, k):
    if l == r:
        return l
    cnt = tree[tree[q].l].cnt - tree[tree[p].l].cnt
    mid = l + r >> 1
    if k <= cnt:
        return query(tree[q].l, tree[p].l, l, mid, k)
    else:
        return query(tree[q].r, tree[p].r, mid + 1, r, k - cnt)
    


def find(x):
    return bisect.bisect_left(nums, x)

idx = 0
tree = [Node() for _ in range(4 * N + N * 17)]
n, m = map(int, input().split())
a = list(map(int, input().split()))
root = [0] * N
nums = sorted(set(a))
root[0] = build(0, len(nums) - 1)
for i in range(1, n + 1):
    root[i] = insert(root[i - 1], 0, len(nums) - 1, find(a[i - 1]))

for _ in range(m):
    l, r, k = map(int, input().split())
    res = query(root[r], root[l - 1], 0, len(nums) - 1, k)
    print(nums[res])

# 5.字典树

## 5.1 01Trie

In [ ]:

class Node:
    __slots__ = 'children', 'cnt'

    def __init__(self):
        self.children = [None, None]
        self.cnt = 0  # 子树大小

class Trie:
    HIGH_BIT = 19

    def __init__(self):
        self.root = Node()

    # 添加 val
    def insert(self, val: int) -> None:
        cur = self.root
        for i in range(Trie.HIGH_BIT, -1, -1):
            bit = (val >> i) & 1
            if cur.children[bit] is None:
                cur.children[bit] = Node()
            cur = cur.children[bit]
            cur.cnt += 1  # 维护子树大小
        return cur

    # 删除 val，但不删除节点
    # 要求 val 必须在 trie 中
    def remove(self, val: int) -> None:
        cur = self.root
        for i in range(Trie.HIGH_BIT, -1, -1):
            cur = cur.children[(val >> i) & 1]
            cur.cnt -= 1  # 维护子树大小
        return cur

    # 返回 val 与 trie 中一个元素的最大异或和
    # 要求 trie 中至少有一个元素
    def query(self, val: int) -> int:
        cur = self.root
        ans = 0
        for i in range(Trie.HIGH_BIT, -1, -1):
            bit = (val >> i) & 1
            # 如果 cur.children[bit^1].cnt == 0，视作空节点
            if cur.children[bit ^ 1] and cur.children[bit ^ 1].cnt:
                ans |= 1 << i
                bit ^= 1
            cur = cur.children[bit]
        return ans

## 5.2字典树

In [ ]:
"""
维护一个字符串集合，支持两种操作：
I x 向集合中插入一个字符串 x
Q x 询问一个字符串在集合中出现了多少次。
"""

from collections import defaultdict


class Node:
    def __init__(self):
        self.children = defaultdict(Node)


class Trie:
    __slots__ = 'root', 'cnt'

    def __init__(self):
        self.root = Node()
        self.cnt = defaultdict(int)

    # 向树中插入字符串
    def insert(self, word: str) -> None:
        cur = self.root
        for w in word:
            cur = cur.children[w]
        self.cnt[cur] += 1  # 出现的次数+1

    # 向树中搜寻字符串
    def search(self, word: str) -> int:
        cur = self.root
        for w in word:
            cur = cur.children.get(w)
            if cur is None:  # 不存在
                return 0
        return self.cnt[cur]  # 存在几个

    # 查询prefix是否是树中字符串的前缀（prefix是否完整的存在树中）
    def startsWith(self, prefix: str) -> bool:
        cur = self.root
        for w in prefix:
            cur = cur.children.get(w)
            if cur is None:
                return False
        return True


n = int(input())
tree = Trie()
for _ in range(n):
    op, s = input().split()
    if op == 'I':
        tree.insert(s)
    else:
        res = tree.search(s)
        print(res)



# 6.树状数组

## 6.1单点修改

In [ ]:
from bisect import *


class BIT:
    def __init__(self, n):
        self.tree = [0] * n  # 注意下标从1开始

    def lowbit(self, x):
        return x & (-x)

    # arr[i] += val
    def update(self, i, val):
        while i < len(self.tree):
            self.tree[i] += val
            i += self.lowbit(i)

    # 返回arr[:i+1]的sum
    def query(self, i):
        res = 0
        while i > 0:
            res += self.tree[i]
            i -= self.lowbit(i)
        return res


# 附上求逆序对板子
n = int(input())
a = list(map(int, input().split()))
b = sorted(set(a))
res = 0
tree = BIT(len(a) + 1)
for x in a:
    i = bisect_right(b, x)  # 比当前元素大的第一个元素，注意这个下标从0开始，而BIT是从1开始的
    res += tree.query(n) - tree.query(i)  # 相减得到比当前大的
    tree.update(i, 1)  # 将索引i处的树状数组值加1
print(res)


## 6.2区间修改

In [ ]:
from collections import defaultdict
class Solution:
    def fullBloomFlowers(self, flowers: List[List[int]], persons: List[int]) -> List[int]:
        res = [0] * len(persons)
        bit = BIT(int(1e9 + 10))
        for l, r in flowers:
            bit.add(l, r, 1)
        for i, p in enumerate(persons):
            res[i] = bit.query(p, p)
        return res


class BIT:
    def __init__(self, n: int):
        self.size = n
        self._tree1 = defaultdict(int)
        self._tree2 = defaultdict(int)

    @staticmethod
    def _lowbit(index: int) -> int:
        return index & -index

    def add(self, left: int, right: int, delta: int) -> None:
        self._add(left, delta)
        self._add(right + 1, -delta)

    def query(self, left: int, right: int) -> int:
        return self._query(right) - self._query(left - 1)

    def _add(self, index: int, delta: int) -> None:
        if index <= 0:
            raise ValueError('index 必须是正整数')

        rawIndex = index
        while index <= self.size:
            self._tree1[index] += delta
            self._tree2[index] += (rawIndex - 1) * delta
            index += self._lowbit(index)

    def _query(self, index: int) -> int:
        if index > self.size:
            index = self.size

        rawIndex = index
        res = 0
        while index > 0:
            res += rawIndex * self._tree1[index] - self._tree2[index]
            index -= self._lowbit(index)
        return res



## 6.3区间包含

In [ ]:
from bisect import *


class BIT:
    def __init__(self, n):
        self.tree = [0] * n  # 注意下标从1开始

    def lowbit(self, x):
        return x & (-x)

    # arr[i] += val
    def update(self, i, val):
        while i < len(self.tree):
            self.tree[i] += val
            i += self.lowbit(i)

    # 返回arr[:i+1]的sum
    def query(self, i):
        res = 0
        while i > 0:
            res += self.tree[i]
            i -= self.lowbit(i)
        return res


# 题目：
# 多少个点包含A而不包含B
# 容斥的思想,包含A的数量-包含A且包含B的数量
n, q = MII()
N = int(2e5 + 10)
tree = BIT(N + 1)
d = [0] * (N + 1)
g = []
for _ in range(n):
    l, r = MII()
    if l > r:
        l, r = r, l
    d[l] += 1
    d[r + 1] -= 1 
    g.append((l, r, 0, 0)) 

for i in range(1, N):  # 包含A点的数量
    d[i] += d[i - 1]

res = [0] * (n + 1)
for i in range(1, q + 1):
    l, r = MII()
    res[i] = d[l]
    if l > r:
        l, r = r, l
    g.append((l, r, i, 1))
# 按端点排序是一个经典的做法，要多思考
g.sort(key=lambda x:(-x[1], x[0], x[3]))  # r由大到小排序。
for i in range(1, n + q + 1):
    l, r, idx, op = g[i - 1]
    if op == 0:
        tree.update(l, 1)  # 插入l
    else:
        res[idx] -= tree.query(l)  # l比当前的小，r又是有序的。说明当前区间被包含了

for i in range(1, q + 1):
    print(res[i])




# 7.有序列表

## 7.1SortedList

In [ ]:
from bisect import bisect_left as lower_bound
from bisect import bisect_right as upper_bound


class FenwickTree:
    def __init__(self, x):
        bit = self.bit = list(x)
        size = self.size = len(bit)
        for i in range(size):
            j = i | (i + 1)
            if j < size:
                bit[j] += bit[i]

    def update(self, idx, x):
        """updates bit[idx] += x"""
        while idx < self.size:
            self.bit[idx] += x
            idx |= idx + 1

    def __call__(self, end):
        """calc sum(bit[:end])"""
        x = 0
        while end:
            x += self.bit[end - 1]
            end &= end - 1
        return x

    def find_kth(self, k):
        """Find largest idx such that sum(bit[:idx]) <= k"""
        idx = -1
        for d in reversed(range(self.size.bit_length())):
            right_idx = idx + (1 << d)
            if right_idx < self.size and self.bit[right_idx] <= k:
                idx = right_idx
                k -= self.bit[idx]
        return idx + 1, k
    
class SortedList:
    block_size = 700

    def __init__(self, iterable=()):
        self.macro = []
        self.micros = [[]]
        self.micro_size = [0]
        self.fenwick = FenwickTree([0])
        self.size = 0
        for item in iterable:
            self.insert(item)

    def insert(self, x):
        i = lower_bound(self.macro, x)
        j = upper_bound(self.micros[i], x)
        self.micros[i].insert(j, x)
        self.size += 1
        self.micro_size[i] += 1
        self.fenwick.update(i, 1)
        if len(self.micros[i]) >= self.block_size:
            self.micros[i:i + 1] = self.micros[i][:self.block_size >> 1], self.micros[i][self.block_size >> 1:]
            self.micro_size[i:i + 1] = self.block_size >> 1, self.block_size >> 1
            self.fenwick = FenwickTree(self.micro_size)
            self.macro.insert(i, self.micros[i + 1][0])

    def pop(self, k=-1):
        i, j = self._find_kth(k)
        self.size -= 1
        self.micro_size[i] -= 1
        self.fenwick.update(i, -1)
        return self.micros[i].pop(j)

    def __getitem__(self, k):
        i, j = self._find_kth(k)
        return self.micros[i][j]

    def count(self, x):
        return self.upper_bound(x) - self.lower_bound(x)

    def __contains__(self, x):
        return self.count(x) > 0

    def lower_bound(self, x):
        i = lower_bound(self.macro, x)
        return self.fenwick(i) + lower_bound(self.micros[i], x)

    def upper_bound(self, x):
        i = upper_bound(self.macro, x)
        return self.fenwick(i) + upper_bound(self.micros[i], x)

    def _find_kth(self, k):
        return self.fenwick.find_kth(k + self.size if k < 0 else k)

    def __len__(self):
        return self.size

    def __iter__(self):
        return (x for micro in self.micros for x in micro)

    def __repr__(self):
        return str(list(self))
    
# 删除的时候要先lower_bound下标再pop

# 8.ST表

In [ ]:
import math
N = 200010
M = 18
f = [[0] * M for _ in range(N)]
n, m =  MII()
a = LII()

for i in range(n):
    f[i][0] = a[i]

def st():  # f[i,j]为以第i个数为起点，长度为2^j的一段区间中的最大值
    for j in range(1, M + 1):
        i = 0
        while i + (1 << j) <= n:   # i - > i+2^j - 1  = 两段 i -> i + 2^(j-1)和i+2^(j-1) - > i+2^j - 1
            f[i][j] = max(f[i][j - 1], f[i + (1 << j - 1)][j - 1])
            i += 1

# 数组下标，查询下标都从0开始
def query(l, r):
    k = int(math.log2(r - l + 1))  # 2 ^ k <= length
    res = max(f[l][k], f[r - (1 << k) + 1][k])  # 可能重叠的两段，下标是0开始的，一段依附l，一段依附r
    return res


st()
for _ in range(m):
    l, r = MII()
    print(query(l, r))